# DermaFace AI — Final Report

**Course:** Mathematics / Deep Learning (Dr. Mahdieh Khalilinezhad) — project in place of the final exam
**Team (7):** Hessam (Product Lead) · Iva (ML Research) · Aparna + Rolando (Data) · Varsha (MLOps) · Temirlan (Eval & Explainability) · Ali (UI/UX)
**Repo:** https://github.com/neuroarcane/dermaface-ai

---

> ### Reading this as a story
> The interesting part of this project isn't the final **macro-F1 of 0.727** — it's *how we got there*:
> success criteria fixed **up front**, a data acquisition that nearly ate two sprints, decisions
> **reversed as evidence arrived** (Keras→PyTorch, severity de-scoped), a from-scratch baseline that
> failed *on purpose* to point at the real bottleneck, and a three-model progression whose **plot twist
> was a fairness gap that only appeared once the models got good.** Sections 0.3 and 7 tell that arc.


## 0. Executive summary

DermaFace AI is an **educational skin-condition screening prototype**. A user uploads a face photo and
receives: (1) a predicted condition — **acne, rosacea, redness, or clear**; (2) a **Grad-CAM** overlay
marking the regions that drove the prediction; and (3) confidence, limitations, and a "see a
professional" prompt. Severity is **de-scoped in v1** (data too sparse — see §3).

**It is framed as screening & education, NOT diagnosis** — a product decision that shapes every UI and
reporting choice, including measuring **fairness across skin tones**.

**Final result:** three models were compared on the same frozen test set; **VGG16 was selected**
(macro-F1 **0.727**, accuracy **0.74**), with an honest, reported **fairness limitation** on the
darkest-skin band.

## 0.1 Success criteria — defined up front

Per the sponsor's guidance we fixed measurable targets **before** modelling, and report actual-vs-target:

| ID | Target |
|---|---|
| **P1** | Beat the majority-class baseline |
| **P2** | Macro-F1 ≥ 0.60 (frozen test set) |
| **P3** | Per-class recall ≥ 0.50 (each class) |
| **Fa1** | Macro-F1 gap across Fitzpatrick skin-tone bands ≤ 0.15 |


## 0.3 Project history — how our thinking evolved

The project did **not** run in a straight line; several core decisions were revised as we learned, and
the schedule was gated almost entirely by **data**.

1. **Framing first.** Fixed measurable success criteria up front (above), so we'd be judged
   target-vs-actual, not vibes.
2. **Data became the critical path — and slipped (the main slowdown).** Most of Fitzpatrick17k's
   original image URLs are **dead**, so only a fraction downloaded directly. We lost roughly a sprint:
   emailed the authors, stood up an **interim MD5-matched Kaggle mirror** (byte-identical) to keep
   moving, then secured the **official copy via the access form**. The license also **forbids hosting
   images in a public repo**, forcing an external-storage workflow. Everything downstream waited on
   this — which is why the early sprints look data-heavy and model-light.
3. **Framework changed: Keras → PyTorch.** The rest of the stack was already PyTorch; we unified rather
   than maintain a split stack.
4. **Scope trimmed on evidence, not opinion.** Severity **de-scoped** (only 6 "severe" / 29 "mild"
   labels); fairness moved to **skin-tone bands** (per-type counts as low as 3); we chose **not** to
   hard-filter to detected faces (would have cut the set to ~293 images).
5. **The modelling story — three models, one plot twist.**
   - The **from-scratch CNN** landed at **macro-F1 0.35 — below baseline.** Not a failure of effort but
     a *diagnosis*: on ~1,000 images a fresh network only learns "clear vs. not-clear." **→ the
     bottleneck is data/representation → borrow pretrained features.**
   - **ResNet50** (transfer learning) nearly **doubled** it to **0.67** and cleared the accuracy bar —
     but introduced the **plot twist**: it **failed the fairness check** (0.25 gap), scoring far worse
     on the darkest-skin band. The weak CNN had "passed" fairness only by being *uniformly* bad; the
     instant a competent model arrived, the gap it had masked appeared. **Getting good is what exposed
     who the tool wasn't good for.**
   - **VGG16** (grid-search-tuned) took the top spot at **macro-F1 0.727** and was **selected**.
6. **What the story adds up to.** The headline isn't "we reached 0.73" — it's that we can **explain
   every number**, including why the fairness gap is a **data-coverage** problem (only 16 dark-skin test
   images), which is why *"collect more dark-skin data"* is our **top future-work item, not an
   afterthought.** The final model ships **with that limitation stated up front.**

## 1. Dataset details

**Business problem.** Acne, rosacea and facial redness are common, but dermatology access is slow and
expensive. The gap is **triage** — helping a person decide *"is this worth getting checked, and roughly
what is it?"* — not diagnosis.

**Datasets (public skin-image collections):**

| Dataset | Role |
|---|---|
| **Fitzpatrick17k** | Condition labels + Fitzpatrick skin-tone labels (fairness) |
| **SKINCON** | Dense clinical concept annotations → severity proxy (de-scoped v1) |
| **Google SCIN** | Consumer/phone-quality photos, closer to deployment |

**Provenance & governance:** official Fitzpatrick17k copy obtained via the authors' access form
(M. Groh); license restricts use to research and **requires deleting the images once research is
complete** (logged as a compliance task) and **forbids public-repo hosting** — so raw images live in
external storage, never committed.

In [ ]:
# --- Exploratory data analysis (EDA) from the cleaned manifest ---
import pandas as pd, matplotlib.pyplot as plt
from dermaface.config import REPO_ROOT, skin_tone_band

df = pd.read_csv(REPO_ROOT / "data/processed/manifest_clean.csv")
df["band"] = df["skin_type"].map(skin_tone_band)
print(f"Cleaned dataset: {len(df)} images across {df['label'].nunique()} classes")

fig, ax = plt.subplots(3, 1, figsize=(6.6, 11))
order = ["acne", "redness", "rosacea", "clear"]
df["label"].value_counts().reindex(order).plot.bar(ax=ax[0], color="#4C72B0")
ax[0].set_title("Class balance (cleaned)"); ax[0].set_ylabel("images"); ax[0].tick_params(axis='x', rotation=0)

df["band"].value_counts().reindex(["I-II","III-IV","V-VI"]).plot.bar(ax=ax[1], color="#55A868")
ax[1].set_title("Fitzpatrick skin-tone bands"); ax[1].set_ylabel("images"); ax[1].tick_params(axis='x', rotation=0)

pd.crosstab(df["label"], df["source"]).reindex(order).plot.bar(stacked=True, ax=ax[2],
    color=["#C44E52", "#8172B3"])
ax[2].set_title("Class x data source"); ax[2].set_ylabel("images"); ax[2].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

### 1.1 The four classes (with examples)

| Class | What it looks like | Total | Main source(s) |
|---|---|---|---|
| **acne** | Comedones, papules, pustules | 623 | Fitzpatrick + SCIN |
| **redness** | Diffuse erythema / inflammation | 471 | Fitzpatrick |
| **rosacea** | Central-face erythema, flushing, telangiectasia | 202 | Fitzpatrick + SCIN |
| **clear** | No target condition | 262 | **SCIN only** |

**⚠️ Source confound (honest limitation).** `clear` is **entirely SCIN** (consumer phone photos) while
the conditions are **mostly Fitzpatrick** (clinical images). A model can score `clear`-vs-not partly by
learning **image *style*, not skin** — which likely inflates the `clear` recall we see in §7. We report
this openly and sanity-check with Grad-CAM that the model attends to skin, not framing.

In [ ]:
# Example images per class (generated locally; NOT committed — Fitzpatrick license forbids public hosting)
from IPython.display import Image as IPyImage, display
from dermaface.config import REPO_ROOT
p = REPO_ROOT / "docs/eval_reports/figures/class_examples.png"
display(IPyImage(str(p), width=660)) if p.exists() else print("class_examples.png not generated on this machine")

## 2. Preprocessing

- **Cleaning:** 1,614 → **1,558** rows (dropped unknown skin type + perceptual-hash duplicates).
- **Class imbalance → weighted loss** (not oversampling): class weights (rosacea ≈ 3.1× acne) applied in
  the loss; the sampler is off so we don't double-correct.
- **Frozen splits** (train/eval/test/demo), re-frozen from cleaned rows with drift ≤ 1 point (seed 42).
- **Augmentation (train only):** crop / flip / rotation + mild brightness/contrast. **Saturation & hue
  are locked at 0 on purpose** — jittering them washes out the erythema that *is* the signal for
  redness/rosacea (enforced by a unit test).

## 3. Model — a three-model bake-off

**Approach:** pretrained backbone + fine-tune, benchmarked against a from-scratch baseline. **Framework:
PyTorch** (unified the stack). Each model is trained on the **same data, splits, and metrics**, then the
best is selected.

| Model | Architecture | Type | Owner |
|---|---|---|---|
| **Basic CNN** | 6-conv-block CNN | From scratch | Ali |
| **ResNet50** | ResNet-50 | Transfer learning (ImageNet) | Iva |
| **VGG16** | VGG-16 | Transfer learning (ImageNet) | Varsha |

**Alternatives rejected:**
- **YOLO / object detection — considered and rejected; *never trained*** (so there are **no YOLO
  hyperparameters** to report). Our task is whole-image **classification** with Grad-CAM localization,
  and our datasets have **no bounding boxes** to train a detector on; YOLO would also make Grad-CAM
  redundant.
- Using the from-scratch CNN as the *product* model — kept only as the baseline.

## 4. Hyperparameter tuning

**Shared configuration (held fixed across models for a fair comparison):**

| Hyperparameter | Value |
|---|---|
| Image size | 224 × 224 |
| **Batch size** | **32** |
| Epochs | 20 |
| Optimizer | Adam |
| Learning rate | 1e-4 |
| Weight decay | 1e-4 |
| Loss | class-weighted cross-entropy |
| Seed | 42 |

**Did we evaluate different batch sizes?** The comparison uses a **fixed batch size of 32** across all
three models *on purpose* — with three architectures to compare on a laptop and a tight deadline, we
held training hyperparameters constant so differences reflect the **architecture**, not the schedule.
The one place we searched was **VGG16**, where Varsha ran a **grid search** over hyperparameters (its
selected configuration is the reported result). A systematic batch-size / LR sweep across all models is
listed as **future work** (§12).

## 5. Training & validation

Each model trains for 20 epochs and checkpoints the **best validation macro-F1** (not the last epoch).
Best-val checkpoints: Basic CNN **0.386 @ epoch 20**, ResNet50 **0.586 @ epoch 20**. **Live
training/validation curves** (per-epoch loss, accuracy, macro-F1) are produced in the per-model training
notebooks in `notebooks/` (`Basic_CNN.ipynb`, `ResNet (1).ipynb`). No over-fitting intervention beyond
early-stopping-on-best-val was needed at 20 epochs.

## 6. Prediction & explainability (Grad-CAM)

`dermaface.inference.predict` runs the full single-image path: preprocess → forward pass → softmax →
prediction **+ Grad-CAM overlay**. The Streamlit app consumes it; the model is a **one-file checkpoint
swap** (`cnn` / `resnet50` / `vgg16`). Below: Grad-CAM on the Basic CNN — the overlay shows *where* the
model looked, so users build appropriate (not blind) trust.

In [ ]:
from IPython.display import Image as IPyImage, display
from dermaface.config import REPO_ROOT
p = REPO_ROOT / "docs/eval_reports/figures/gradcam_demo.png"
display(IPyImage(str(p), width=660)) if p.exists() else print("gradcam_demo.png not generated on this machine")

## 7. Evaluation

All models are scored the **same way** on the **same frozen test set** (158 images) via
`dermaface.training.metrics`. Selection metric = **macro-F1** (chosen up front for the class imbalance),
with per-class recall and the fairness gap as guards.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

BASELINE_ACC = 0.3987
res = pd.DataFrame([
    {"Model":"Basic CNN (baseline)", "Accuracy":0.348, "Macro-F1":0.354,
     "acne":0.06,"rosacea":0.45,"redness":0.42,"clear":0.84, "Fairness gap":0.064, "Selected":""},
    {"Model":"ResNet50 (Iva)", "Accuracy":0.677, "Macro-F1":0.674,
     "acne":0.48,"rosacea":0.77,"redness":0.77,"clear":0.92, "Fairness gap":0.252, "Selected":""},
    {"Model":"VGG16 (Varsha)", "Accuracy":0.740, "Macro-F1":0.727,
     "acne":None,"rosacea":None,"redness":None,"clear":None, "Fairness gap":None, "Selected":"✅"},
])
print("Per-class cells are recall; VGG16 per-class/fairness not in Varsha's summary (aggregate only).")
display(res)

fig, ax = plt.subplots(figsize=(6.6,4))
ax.bar(res["Model"], res["Macro-F1"], color=["#B0B0B0","#4C72B0","#55A868"])
ax.axhline(0.60, ls="--", color="crimson", label="P2 target (0.60)")
ax.axhline(BASELINE_ACC, ls=":", color="gray", label="majority baseline acc (0.40)")
ax.set_ylabel("Macro-F1"); ax.set_title("Model comparison — macro-F1 (frozen test set)")
ax.legend(); plt.xticks(rotation=10); plt.tight_layout(); plt.show()

In [ ]:
# Fairness by skin-tone band (available for CNN and ResNet50; VGG16 per-band not reported yet)
import matplotlib.pyplot as plt, numpy as np
bands = ["I-II","III-IV","V-VI"]
cnn_b  = [0.333, 0.367, 0.304]
res_b  = [0.665, 0.690, 0.438]
x = np.arange(len(bands)); w = 0.38
fig, ax = plt.subplots(figsize=(6.6,4))
ax.bar(x-w/2, cnn_b, w, label="Basic CNN (gap 0.064)", color="#B0B0B0")
ax.bar(x+w/2, res_b, w, label="ResNet50 (gap 0.252)", color="#4C72B0")
ax.set_xticks(x); ax.set_xticklabels(bands); ax.set_ylabel("Macro-F1")
ax.set_title("Fairness by Fitzpatrick skin-tone band")
ax.legend(); plt.tight_layout(); plt.show()
print("The competent model (ResNet50) improves most on I–IV but barely on V–VI (n=16) → the gap WIDENS to 0.25.")

### 7.1 False positives / false negatives — causes and treatment (Basic CNN)

Reading the confusion matrix, errors fall into three groups, ordered by how much they matter for a
**screening** tool:

1. **Condition → "clear" (false negatives — most harmful).** A person with a condition told nothing's
   there (false reassurance). *Treatment (product):* the app never says "you're fine" — every screen ends
   in "see a professional"; low confidence is shown. *Treatment (model):* pretrained backbone + more
   data; a confidence threshold routing uncertain cases to *"inconclusive."*
2. **Condition ↔ condition confusion.** acne, rosacea and redness are all erythematous and visually
   adjacent. *Treatment:* transfer learning (helped — §7); if fine separation stays hard, fall back to a
   coarser *"inflammatory vs. clear"* decision or a top-2 output.
3. **"clear" → condition (false positives — least harmful).** In screening this errs the *safe* way
   (toward "get checked"). *Treatment:* calibrate the threshold; Grad-CAM shows the (weak) evidence.

Real misclassified examples (generated locally; not committed — license):

In [ ]:
from IPython.display import Image as IPyImage, display
from dermaface.config import REPO_ROOT
for name in ["fn_examples.png", "fp_examples.png"]:
    p = REPO_ROOT / "docs/eval_reports/figures" / name
    display(IPyImage(str(p), width=660)) if p.exists() else print(name, "not generated on this machine")

### 7.2 Model selection — the progression and the final pick

1. **Baseline** (from-scratch CNN) → **macro-F1 0.35, fails every target** → bottleneck is
   data/representation, not effort.
2. **ResNet50** → **0.67**, clears accuracy, **but fails fairness (0.25 gap).**
3. **VGG16** → **0.727**, the best macro-F1 → **selected.**

**Final selection: ✅ VGG16** (accuracy 0.74, macro-F1 0.727). *Why over ResNet50:* higher macro-F1 and
accuracy on the same frozen test set. *Why not the CNN:* it fails every performance target.

**Two honest asterisks (kept visible):**
1. **VGG16's fairness gap is not yet in hand** — Varsha reported aggregate metrics only; the per-band
   number lives in her `vgg16_eval_report.docx`. Since ResNet50's 0.25 gap is a **shared data-coverage**
   problem, VGG16 likely shares it; we report the actual number, not an assumption.
2. **App swap + plates** for VGG16 land once `dermaface_best_vgg16.pt` is shared (the factory already
   supports `vgg16`).

## 8. Fairness analysis

We report fairness across **skin-tone bands (I-II / III-IV / V-VI)** because per-Fitzpatrick-type test
counts are as low as 3 (type VI is ~2% of the data). Results (§7): Basic CNN gap **0.064** ✅, ResNet50
gap **0.252** ❌.

**The key insight (reported, not hidden):** the CNN *passed* fairness only because it was **uniformly
bad** — a small gap between three low scores is incompetence spread evenly, not fairness. As soon as a
**competent** model arrived, the gap **widened to 0.25**: its gains landed on the well-represented
lighter-skin bands while **V–VI barely improved.** *Model skill and the fairness gap grew together.*
**Root cause is data coverage** (only 16 type-V/VI test images), so the remediation is **more dark-skin
data (e.g. DDI)** — reported as required future work, and the final model ships with the gap stated as a
headline limitation.

## 10. Interpretation of results

For a **screening** tool (not diagnosis): the safe error is a false *positive* (tells the user to get
checked when they're fine); the dangerous error is a false *negative* (misses a condition). The product
is designed so the model's weaknesses fail safe — it never issues an all-clear, always defers to a
professional, and shows confidence + Grad-CAM so the user can judge the evidence. The **fairness gap** is
the most important real-world caveat: the tool is currently **less reliable on the darkest skin tones**,
which we state plainly rather than bury.

In [ ]:
# 11. Hardware & memory / reproducibility
import platform, multiprocessing, torch
print("Platform :", platform.platform())
print("CPU cores:", multiprocessing.cpu_count())
print("PyTorch  :", torch.__version__)
print("Device   : trained on CPU (CUDA:", torch.cuda.is_available(),
      "| MPS:", torch.backends.mps.is_available(), ")")
print("Reproducibility: fixed seed 42, frozen test split, best-val-macro-F1 checkpointing.")

## 12. Next steps

**Immediate (Sprint 4 → submission):** presentation practice (Tue/Wed 5pm) · record (Thu) · **submit
report (Fri).**

**Future work:** (1) **More dark-skin data (DDI)** to close the fairness gap — top priority; (2) a
systematic **LR / batch-size / augmentation sweep** across all models; (3) mix data sources per class to
remove the `clear`=SCIN **source confound**; (4) revisit **severity** with a concept-derived proxy once
labels are denser; (5) clinician feedback.

## 13. Lessons learned

- **Data was the critical path — and we under-estimated it.** Dead URLs + licensing cost ~a sprint and
  gated everything. What worked: an interim MD5-matched mirror to unblock training, and treating data as
  a first-class workstream.
- **Baseline first pays off.** The deliberately-weak CNN told us the bottleneck is data/representation,
  justifying transfer learning **with evidence**.
- **Decide from the data.** Severity de-scope, fairness-by-bands, and not-face-filtering were settled by
  **counting the data**, not opinion.
- **Competence can *expose* a fairness gap, not just create one.** Our weakest model "passed" fairness by
  being evenly bad; our best model revealed a 0.25 dark-skin gap. A single headline metric hides *who* a
  model works for — measuring per band from the outset is what caught it. **The most important thing we
  learned.**
- **Task-appropriate over fanciest.** YOLO was evaluated on paper and rejected, never trained.

## 14. Individual contributions

| Member | Contributions |
|---|---|
| **Hessam** (Product Lead) | Scope, disclaimer/ethics framing, Day-1 report, coordination; report editing |
| **Iva** (ML Research) | Backbone decision, severity method, metrics implementation + tests; **trains ResNet50**; report editing |
| **Aparna + Rolando** (Data) | Data acquisition + cleaning: manifest, dedup + skin-type validation, weighted-loss imbalance, frozen splits, augmentation, QA + fairness-coverage findings; provenance |
| **Varsha** (MLOps) | Training/MLOps infra (loop, checkpointing, CI); **trains VGG16 (grid search, selected model)**; HF deploy |
| **Temirlan** (Eval & Explainability) | Evaluation + failure-analysis support; Grad-CAM evidence; metrics test support |
| **Ali** (UI/UX) | Streamlit app (upload/consent/disclaimer/UI states) + Grad-CAM display; **trains the Basic CNN baseline + eval report**; wired real inference + Grad-CAM into the app; standups/sprint tracking; severity-decision write-up |

---

*Data note: dataset images shown in this notebook are rendered from local research data and are **not**
committed to the public repository (Fitzpatrick17k license). This report is the consolidated view; the
per-model training notebooks and the app live in the repo.*